# Validação do Sensor Virtual — Backflow Recife
Execute de cima para baixo. Começa com **dados sintéticos** só para testar o pipeline; troque `FONTE` para `"real"` quando tiver o log e as ocorrências. Leia `README_PROTOCOLO.md` antes: as metas e o desenho da análise devem ser fixados **antes** de olhar o período de teste.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sensor_stats as ss
warnings.filterwarnings("ignore")

FONTE = "sintetico"   # troque para "real"
CAMINHOS = {
    "sintetico": ("sintetico/sintetico_log.csv", "sintetico/sintetico_ocorrencias.csv", 300),
    "real": ("../data/leituras_log.csv", "ocorrencias.csv", 30),
}
LOG, OCORR, INTERVALO_S = CAMINHOS[FONTE]
TABUA = "../data/mare_recife.csv"
JANELA, LAG = "1h", "0h"       # janela de análise e antecedência da previsão
BLOCO, B = 24, 2000            # bloco (em janelas) e reamostragens do bootstrap
NIVEL_ALERTA = 2               # 2 = potencial ou pior; 3 = só ocorrendo
OUT = "resultados"; os.makedirs(OUT, exist_ok=True)
if FONTE == "sintetico" and not os.path.exists(LOG):
    import gerar_sinteticos; gerar_sinteticos.gerar()
if FONTE == "sintetico":
    print("*** DADOS SINTETICOS: apenas teste do pipeline. NAO usar no artigo. ***")

## 1. Dados e qualidade

In [ ]:
tabua = ss.carregar_tabua(TABUA)
log = ss.carregar_log(LOG)
oc = ss.carregar_ocorrencias(OCORR)
log = ss.adicionar_mare_interpolada(log, tabua)
display(pd.Series(ss.relatorio_qualidade(log, INTERVALO_S)))
print("ocorrencias:", len(oc), "| eventos (apos agrupar em 6 h):", len(ss.agrupar_eventos(oc["inicio"])))

## 2. Janelas e rótulos
`w`: janelas com a maré interpolada (cosseno) entre eventos da tábua, como o sensor calcula; limiares em metros.

In [ ]:
w = ss.construir_janelas(log, oc, JANELA, LAG, INTERVALO_S)
print(f"janelas: {len(w)} | positivas: {int(w['y'].sum())} ({w['y'].mean():.2%})")
display(pd.Series(ss.checar_amostra(w["y"])))

## 3. Desempenho do sensor (IC por bootstrap em blocos; POD com Clopper-Pearson)

In [ ]:
res = {}
for nome, ww in (("sensor", w),):
    tab, c = ss.tabela_metricas(ww["y"], ss.yhat_sensor(ww, NIVEL_ALERTA), BLOCO, B)
    res[nome] = tab
    print(nome, c); display(tab.round(3))
pd.concat(res, axis=1).to_csv(f"{OUT}/metricas_sensor.csv")

## 4. Nível de evento e antecedência

In [ ]:
ev = ss.avaliar_eventos(w, oc["inicio"], NIVEL_ALERTA, janela=JANELA)
display(pd.Series(ev))

## 5. Calibração de metas: grade chuva × maré (mapa de calor do TSS; maré em metros)

In [ ]:
chuvas = np.unique(np.round(w["chuva_max"].dropna().quantile([.5, .7, .8, .9, .95, .98]), 1))
mares = np.round(np.arange(1.0, 2.8, 0.2), 2)
g = ss.grade_limiares(w, chuvas, mares, beta=2.0)
g.to_csv(f"{OUT}/grade_limiares.csv", index=False)
piv = g.pivot(index="chuva_mm", columns="mare_m", values="TSS")
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(piv.values, origin="lower", aspect="auto", cmap="viridis")
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
ax.set_xlabel("limiar de mare (m)"); ax.set_ylabel("meta de chuva (mm)")
fig.colorbar(im, label="TSS"); plt.show()
display(g.sort_values("TSS", ascending=False).head(5)[["chuva_mm", "mare_m", "POD", "FAR", "CSI", "TSS", "Fbeta"]].round(3))

## 6. Validação temporal (calibra no passado, testa no futuro com metas congeladas)

In [ ]:
corte = w.index[int(len(w) * 0.6)]
r = ss.selecionar_e_testar(w, corte, chuvas, mares)
print("metas escolhidas no treino:", r["chuva_mm"], "mm /", r["mare_m"], "| eventos no teste:", r["eventos_teste"])
display(pd.Series(r["teste"]).round(3))
ro = ss.rolling_origin(w, chuvas, mares, n_folds=4)
ro.to_csv(f"{OUT}/rolling_origin.csv", index=False)
display(ro.round(3))

## 7. A maré importa? Linhas de base, interação (Firth) e comparação pareada

In [ ]:
c0 = np.nanquantile(w["chuva_max"], 0.9)
base = {
    "so_chuva": (w["chuva_max"] >= 5).to_numpy(),
    "so_mare": (w["mare_h"] >= 2.0).to_numpy(),
    "chuva_e_mare": ((w["chuva_max"] >= 5) & (w["mare_h"] >= 2.0)).to_numpy(),
    "sensor_potencial": ss.yhat_sensor(w, 2),
}
y = w["y"].to_numpy()
tb = pd.DataFrame({k: ss.metricas(y, v) for k, v in base.items()}).T[["TP", "FP", "FN", "POD", "FAR", "CSI", "TSS"]]
display(tb.round(3)); tb.to_csv(f"{OUT}/baselines.csv")

tab, lrt, p_int = ss.teste_interacao(w)
display(tab.round(3)); print(f"LRT interacao chuva x mare: {lrt:.2f}  p = {p_int:.4f}")

prob_i = ss.prob_temporal(w, True); prob_p = ss.prob_temporal(w, False)
ok = ~np.isnan(prob_i)
auc = {"so_chuva": ss.auc_roc_pr(y[ok], w["chuva_max"].to_numpy()[ok]),
       "so_mare": ss.auc_roc_pr(y[ok], w["mare_h"].to_numpy()[ok]),
       "logit_sem_interacao": ss.auc_roc_pr(y[ok], prob_p[ok]),
       "logit_com_interacao": ss.auc_roc_pr(y[ok], prob_i[ok])}
display(pd.DataFrame(auc, index=["AUC_ROC", "AUC_PR"]).T.round(3))
est, lo, hi = ss.auc_diferenca_bloco(y[ok], prob_i[ok], w["chuva_max"].to_numpy()[ok], BLOCO)
print(f"AUC(logit c/ interacao) - AUC(so chuva) = {est:.3f}  IC95 [{lo:.3f}, {hi:.3f}]")

cp = ss.comparar_pareado(y, base["sensor_potencial"], base["so_chuva"])
display(pd.DataFrame(cp).T)
p_adj = ss.holm([p_int] + [v["p_mcnemar_exato"] for v in cp.values()])
print("p ajustados (Holm): interacao, deteccao, falsos alarmes ->", np.round(p_adj, 4))

## 8. Risco relativo e calibração probabilística

In [ ]:
rr = ss.risco_relativo(w, 5.0, 0.75); display(pd.Series(rr, dtype=object))
tab_cal, brier, bss = ss.calibracao(y, prob_i)
display(tab_cal.round(3)); print(f"Brier = {brier:.4f} | Brier skill score vs climatologia = {bss:.3f}")

## 9. Robustez: janela e antecedência (TSS e CSI do sensor)

In [ ]:
linhas = []
for j in ("30min", "1h", "3h"):
    for lg in ("0h", "1h", "3h"):
        ww = ss.construir_janelas(log, oc, j, lg, INTERVALO_S)
        m = ss.metricas(ww["y"], ss.yhat_sensor(ww, NIVEL_ALERTA))
        linhas.append({"janela": j, "lag": lg, "eventos": int(ww["y"].sum()), "POD": m["POD"], "FAR": m["FAR"], "CSI": m["CSI"], "TSS": m["TSS"]})
rob = pd.DataFrame(linhas); rob.to_csv(f"{OUT}/robustez.csv", index=False)
display(rob.round(3))

## 10. Tamanho amostral
Quantos eventos para um IC95 de largura 0,20 em torno de um POD de 0,8?

In [ ]:
print("eventos necessarios:", ss.n_para_ic(0.8, 0.20))
print("eventos observados :", ss.checar_amostra(w["y"]))